In [66]:
import pandas as pd
import pyarrow.feather as feather
import numpy as np
#feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step18.ftr"
# "please take the file: glm_data_step2_final.ftr and give it to Gary."

import os as os
#import matplotlib.pyplot as plt
import time as time
import random



In [2]:
"""1)	Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 
2)	Read “necessary columns” from step 18 feather file since there will be memory issue otherwise. a.	“necessary columns” = columns needed for objective function, VIN, Date
b.	Merge this data with “superpolicy_final.sas7bdat” using VIN-Date 
    i.	Confirm the number of extra rows after the merge
    1.	Ideally there wont be any extra rows 
        i. Rows were not same. 
3)	Loop to find ideal fits run 500 simulation. 
    a.	For each loop get the 24D vector 
    b.	Calculate the Objective function for train, valid, holdout. 
    c.	If the error is less than SD, then proceed in the loop
    d.	Store at end of loop, seed, Objective function, time taken for each loop 
4)	Plot Loop vs Objective function. 
    Plot Hist of Objective function. 
    a.Choose lowest Objective function ? """




'1)\tOpen “superpolicy_final.sas7bdat” . a.\t This file has the superpolicy_id based on the logic Gary wrote in SAS i.\tWhere is that file (store for completion) \n2)\tRead “necessary columns” from step 18 feather file since there will be memory issue otherwise. a.\t“necessary columns” = columns needed for objective function, VIN, Date\nb.\tMerge this data with “superpolicy_final.sas7bdat” using VIN-Date \ni.\tConfirm the number of extra rows after the merge\n1.\tIdeally there wont be any extra rows \n3)\tLoop to find ideal fits run 100 simulation. \na.\tFor each loop get the 24D vector \nb.\tCalculate the Objective function. \nc.\tIf the error is less than SD, then proceed in the loop\nd.\tStore following loop, seed, Objective function, time taken for each loop \n4)\tPlot Loop vs Objective function. \na.\tChoose lowest Objective function ? '

- Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 

In [3]:
base_path = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/'

full_feather_file_path = os.path.join(base_path,  "glm_data_step18.ftr")
columns_file_path = os.path.join(base_path,  "columns_glm_data_step2_final_feather.csv") 
superpolicy_file =  os.path.join(base_path, 'superpolicy_final.sas7bdat')


""" 
feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.ftr"

csv_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_1000row_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_allrow_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.csv"


file_df1_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df1_1_superpolicy.csv"

file_df2_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df2_1_superpolicy.csv" """


' \nfeather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.ftr"\n\ncsv_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"\ncsv_1000row_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"\ncsv_allrow_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.csv"\n\n\nfile_df1_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df1_1_superpolicy.csv"\n\nfile_df2_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df2_1_superpolicy.csv" '

In [4]:
df_a = pd.read_sas( superpolicy_file ,format = 'sas7bdat', encoding="ISO-8859-1")  # read sas7bdat file


In [19]:
df_a['Date'] = pd.to_datetime(df_a['Date'])
df_a['DateYMD'] = df_a['Date'].dt.strftime('%Y%m%d')
df_a['VIN_Date'] = df_a['VIN'] +'_' +  df_a['DateYMD']


In [21]:
df_a.drop(columns = ['DateYMD','VIN','Date'], inplace = True)

In [ ]:
df_a = df_a.drop_duplicates( subset = 'VIN_Date', keep = 'first')

In [22]:
df_a.head()

,ID,pol,superpolicy_id,VIN_Date
0,1.0,EAA1345937,1.0,1J4GK48K42W273250_20170101
1,2.0,EAA1347517,2.0,1FMYU03122KA07434_20170101
2,3.0,EAA1347517,2.0,1FMYU03122KA07434_20180101
3,4.0,EAA1347848,4.0,2GTEC13V361342050_20170101
4,5.0,EAA1347848,4.0,2GTEC13V361342050_20180103


In [68]:
SeedNum = 1
unique_values = df_a['superpolicy_id'].unique().tolist()


In [72]:
np.random.seed(SeedNum)  
shuffleunique_values = random.sample(unique_values, len(unique_values))
temp = sum([a -b for a,b in zip(unique_values, shuffleunique_values) ])

print(temp)

0.0


In [74]:
unique_values = df_a['superpolicy_id'].unique().tolist()

In [82]:
#Experiment to confirm that random seed is reproducible. 
# Lesson: random seed and np.random seed are different things. 
# Since we are using random.sample since that is the only way to shuffle without altering original copy. 
# np.shuffle was changing original - that is , it was in place. 
if 0:
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)

114610163157152.0
114690651502996.0
114624283447906.0
114667652701702.0
114676522262654.0
114687728426810.0
114608769377382.0
114575389504216.0
114671046290822.0


In [47]:
def get_tvh(df_a,SeedNum,unique_values,tv_split_colname):
    
    TrainPercentage = 0.6
    ValidPercentage = 0.2

    np.random.seed(SeedNum)  
    shuffleunique_values = random.sample(unique_values, len(unique_values))

    total_unique = len(unique_values)
    train_end = int(total_unique * TrainPercentage)
    val_end = train_end + int(total_unique * ValidPercentage)

    train_values = unique_values[:train_end]
    val_values = unique_values[train_end:val_end]
    holdout_values = unique_values[val_end:]

    value_to_split = {}
    for value in train_values:
        value_to_split[value] = 'train'
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)
    return df_a

In [86]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_coll', 'ee_comp','ee_med',
       'incurred_raw_bi', 'incurred_raw_pd', 'incurred_raw_pip', 
       'incurred_raw_coll','incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [87]:
#db_ppee  
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp']
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )
#write a function 
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']

In [190]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file.reset_index()
stat_file

,Unnamed: 0,mean_pp,SDev,mean_ee
0,CAR_pp_bi,147.732876,2.145207,2.504728e+06
1,CAR_pp_pd,109.313637,0.624208,2.525973e+06
2,CAR_pp_pip,94.588838,4.274305,7.547369e+05
3,CAR_pp_med,29.121543,0.533489,1.844539e+06
4,CAR_pp_coll,211.947742,0.984054,1.868875e+06
5,CAR_pp_comp,90.881698,0.530966,1.962455e+06
6,SUV_pp_bi,128.235191,2.581962,1.739631e+06
7,SUV_pp_pd,105.625418,0.712025,1.746269e+06
8,SUV_pp_pip,84.862907,6.035058,5.323518e+05
9,SUV_pp_med,28.191644,0.519492,1.326795e+06


In [196]:
def objectivefunction(train_df_a_pp,stat_file):
    mycol = train_df_a_pp.columns
    stat_file['sample'] = train_df_a_pp[mycol]
    stat_file['norm_mean_ee'] = stat_file['mean_ee']/np.sum(stat_file['mean_ee'])
    objfunval_train_df = np.sum(np.abs(stat_file['sample'] - stat_file['mean_pp'])*stat_file['norm_mean_ee'])
    return objfunval_train_df


In [200]:
def call_objectivefunction(train_df_a,df1,pp_columns):
    #merge with pp, ee db 
    train_df_a = pd.merge(train_df_a, df1, on = 'VIN_Date', how = 'left')
    train_df_a_subset_flattened = generate_row(train_df_a)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    #send to objective function 
    train_objval = objectivefunction(train_df_a_pp,stat_file)
    return train_objval

In [201]:
train_objval = call_objectivefunction(train_df_a,df1,pp_columns)

In [203]:
start_time = time.time()
unique_values = df_a['superpolicy_id'].unique()
tv_split_colname = 'tv_split' #+str(SeedNum)
within_sd_list = []
outof_sd_list = []
pp_columns = stat_file['Unnamed: 0'] # [column for column in stat_file.columns if 'pp' in column]

listtotal_obj_fun  = []
#vSeedNum = 2#42
metconditon = 1
for SeedNum in range(0,1):

    #get tv_split in df_a
    df_a = get_tvh(df_a,SeedNum,unique_values,tv_split_colname)

    # Get train, text, val 
    train_df_a =  df_a.loc[df_a['tv_split'] == 'train'].reset_index().drop(['index','tv_split'], axis=1)
    valid_df_a =  df_a.loc[df_a['tv_split'] == 'valid'].reset_index().drop(['index','tv_split'], axis=1)
    holdout_df_a =  df_a.loc[df_a['tv_split'] == 'holdout'].reset_index().drop(['index','tv_split'], axis=1)

    train_objval = call_objectivefunction(train_df_a,df1,pp_columns)
    valid_objval = call_objectivefunction(valid_df_a,df1,pp_columns)
    holdout_objval = call_objectivefunction(holdout_df_a,df1,pp_columns)

    total_obj_fun = (train_objval + valid_objval + holdout_objval)/3
    listtotal_obj_fun.append(total_obj_fun)  
    #Check condition 
    metconditon = 1
    if metconditon:
        within_sd_list.append(SeedNum) 
    else: 
        outof_sd_list.append(SeedNum) 
    #calculate train, valid, holdout Objective function
    print(df_a[tv_split_colname].value_counts())

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time : {elapsed_time : .2f} seconds")



tv_split
train      43205181
valid      14372309
holdout    14196127
Name: count, dtype: int64
Elapsed time :  706.88 seconds


In [207]:
holdout_objval

1.3151249755863654

In [114]:
 #merge with pp, ee db 
    train_df_a = pd.merge(train_df_a, df1, on = 'VIN_Date', how = 'left')
    train_df_a_subset_flattened = generate_row(train_df_a)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    #send to objective function 
    train_objval = objectivefunction(train_df_a_pp,stat_file)


In [110]:
pp_columns

['mean_pp']

In [10]:
#Read feather file columns
if 0:
    table = feather.read_table(full_feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)
    df_columns.shape
    df_columns.nunique()
    duplicates = df_columns[df_columns.duplicated(keep= False)]
    print(duplicates)   


In [13]:
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','ee_um_uim','ee_tot','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','incurred_raw_um_uim','incurred_raw_tot']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']

In [34]:
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )

In [35]:
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']

In [43]:
df1.shape

(67451697, 26)

In [37]:
df1.head()

,VIN,Date,ee_bi,ee_pd,ee_pip,ee_med,ee_coll,ee_comp,vc_Vehicle_Type,incurred_raw_bi,...,incurred_raw_coll,incurred_raw_comp,claim_count_bi,claim_count_pd,claim_count_pip,claim_count_med,claim_count_coll,claim_count_comp,DateYMD,VIN_Date
0,137FA90382E201482,2020-04-09,0.342,0.342,0.342,0.342,0.342,0.342,SUV,0,...,0,0,0,0,0,0,0,0,20200409,137FA90382E201482_20200409
1,1C4BJWDG5HL554922,2020-11-14,0.003,0.003,0.003,0.003,0.003,0.003,SUV,0,...,0,0,0,0,0,0,0,0,20201114,1C4BJWDG5HL554922_20201114
2,1C4BJWEG0EL278398,2020-05-24,0.999,0.999,0.000,0.999,0.999,0.999,SUV,0,...,0,0,0,0,0,0,0,0,20200524,1C4BJWEG0EL278398_20200524
3,1C4BJWEG3FL657980,2020-12-12,0.999,0.999,0.000,0.999,0.999,0.999,SUV,0,...,0,0,0,0,0,0,0,0,20201212,1C4BJWEG3FL657980_20201212
4,1C4NJCEA2GD700319,2020-02-25,1.002,1.002,1.002,1.002,1.002,1.002,SUV,0,...,0,0,0,0,0,0,0,0,20200225,1C4NJCEA2GD700319_20200225


In [39]:
df1 = pd.merge(df1, df_a, on = 'VIN_Date', how = 'left')

In [40]:
df1['VIN_Date'].nunique()

67451697

In [44]:
df1.shape

(67451697, 26)

In [41]:
df_a['VIN_Date'].nunique()

71773617

In [42]:
df1.shape

(67451697, 26)

In [38]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_med', 'ee_coll', 'ee_comp',
       'claim_count_bi', 'claim_count_pd', 'claim_count_pip',
       'claim_count_coll', 'claim_count_comp', 'claim_count_med','incurred_raw_bi',
       'incurred_raw_pd', 'incurred_raw_pip', 'incurred_raw_coll',
       'incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [42]:
df1_group_flattened = generate_row(df1)

In [45]:
from itertools import chain 
ee_columns = [column for column in df1_group_flattened.columns if 'ee' in column]
pp_columns = [column for column in df1_group_flattened.columns if 'pp' in column]
all_columns = list(chain(pp_columns,ee_columns))
print(all_columns)

['CAR_pp_bi', 'CAR_pp_pd', 'CAR_pp_pip', 'CAR_pp_med', 'CAR_pp_coll', 'CAR_pp_comp', 'SUV_pp_bi', 'SUV_pp_pd', 'SUV_pp_pip', 'SUV_pp_med', 'SUV_pp_coll', 'SUV_pp_comp', 'TRUCK_pp_bi', 'TRUCK_pp_pd', 'TRUCK_pp_pip', 'TRUCK_pp_med', 'TRUCK_pp_coll', 'TRUCK_pp_comp', 'VAN_pp_bi', 'VAN_pp_pd', 'VAN_pp_pip', 'VAN_pp_med', 'VAN_pp_coll', 'VAN_pp_comp', 'CAR_ee_bi', 'CAR_ee_pd', 'CAR_ee_pip', 'CAR_ee_med', 'CAR_ee_coll', 'CAR_ee_comp', 'SUV_ee_bi', 'SUV_ee_pd', 'SUV_ee_pip', 'SUV_ee_med', 'SUV_ee_coll', 'SUV_ee_comp', 'TRUCK_ee_bi', 'TRUCK_ee_pd', 'TRUCK_ee_pip', 'TRUCK_ee_med', 'TRUCK_ee_coll', 'TRUCK_ee_comp', 'VAN_ee_bi', 'VAN_ee_pd', 'VAN_ee_pip', 'VAN_ee_med', 'VAN_ee_coll', 'VAN_ee_comp']


In [48]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file.head()

,Unnamed: 0,mean_pp,SDev,mean_ee
0,CAR_pp_bi,147.732876,2.145207,2.504728e+06
1,CAR_pp_pd,109.313637,0.624208,2.525973e+06
2,CAR_pp_pip,94.588838,4.274305,7.547369e+05
3,CAR_pp_med,29.121543,0.533489,1.844539e+06
4,CAR_pp_coll,211.947742,0.984054,1.868875e+06


In [53]:
stat_file.index = stat_file['Unnamed: 0']

In [57]:
stat_file.drop(columns = 'Unnamed: 0')

,mean_pp,SDev,mean_ee
Unnamed: 0,,,
CAR_pp_bi,147.732876,2.145207,2.504728e+06
CAR_pp_pd,109.313637,0.624208,2.525973e+06
CAR_pp_pip,94.588838,4.274305,7.547369e+05
CAR_pp_med,29.121543,0.533489,1.844539e+06
CAR_pp_coll,211.947742,0.984054,1.868875e+06
CAR_pp_comp,90.881698,0.530966,1.962455e+06
SUV_pp_bi,128.235191,2.581962,1.739631e+06
SUV_pp_pd,105.625418,0.712025,1.746269e+06
SUV_pp_pip,84.862907,6.035058,5.323518e+05


In [49]:
df1_24D = df1_group_flattened[all_columns]

In [51]:
df1_24D.head()

,CAR_pp_bi,CAR_pp_pd,CAR_pp_pip,CAR_pp_med,CAR_pp_coll,CAR_pp_comp,SUV_pp_bi,SUV_pp_pd,SUV_pp_pip,SUV_pp_med,...,TRUCK_ee_pip,TRUCK_ee_med,TRUCK_ee_coll,TRUCK_ee_comp,VAN_ee_bi,VAN_ee_pd,VAN_ee_pip,VAN_ee_med,VAN_ee_coll,VAN_ee_comp
0,146.712995,108.805174,94.300129,28.71039,210.535093,90.010657,127.55712,105.123274,84.6556,27.856535,...,1253255.5,5354148.0,4166788.0,4396414.0,1356266.875,1362143.375,410796.40625,1069605.875,936386.9375,989668.8125


,CAR_ee_bi,CAR_ee_pd,CAR_ee_pip,CAR_ee_med,CAR_ee_coll,CAR_ee_comp,CAR_claim_count_bi,CAR_claim_count_pd,CAR_claim_count_pip,CAR_claim_count_coll,...,VAN_incurred_raw_pip,VAN_incurred_raw_coll,VAN_incurred_raw_comp,VAN_incurred_raw_med,VAN_pp_bi,VAN_pp_pd,VAN_pp_pip,VAN_pp_med,VAN_pp_coll,VAN_pp_comp
0,12459648.0,12565764.0,3761403.5,9161083.0,9303832.0,9769706.0,81811.0,334967.0,35191.0,479193.0,...,41854140.0,133351660.0,76249909.0,23693471.0,113.319737,91.211024,101.885361,22.15159,142.41085,77.045884


In [ ]:
df2_1_stat_pp_N500.csv

In [ ]:
df_d = pd.read_csv(os.path.join(fast_eda_path, 'df_design.csv'))

In [7]:
np.max(df_a['superpolicy_id'])

71969635.0

In [8]:
Num_unique = df_a['superpolicy_id'].nunique()


In [9]:
unique_values = df_a['superpolicy_id'].unique()

In [13]:


start_time = time.time()
#vSeedNum = 2#42
for SeedNum in range(1,6):
    tv_split_colname = 'tv_split_'+str(SeedNum)
    TrainPercentage = 0.6
    ValidPercentage = 0.2

    np.random.seed(SeedNum)  
    np.random.shuffle(unique_values)

    total_unique = len(unique_values)
    train_end = int(total_unique * TrainPercentage)
    val_end = train_end + int(total_unique * ValidPercentage)

    train_values = unique_values[:train_end]
    val_values = unique_values[train_end:val_end]
    holdout_values = unique_values[val_end:]

    value_to_split = {}
    for value in train_values:
        value_to_split[value] = 'train'
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)

    print(df_a[tv_split_colname].value_counts())

    end_time = time.time()
    elapsed_time = end_time - start_time

print(f"Elapsed time : {elapsed_time : .2f} seconds")


tv_split_1
train      42887589
valid      14821519
holdout    14260528
Name: count, dtype: int64
tv_split_2
train      42866402
valid      14773271
holdout    14329963
Name: count, dtype: int64
tv_split_3
train      42990640
holdout    14676756
valid      14302240
Name: count, dtype: int64
tv_split_4
train      42866107
valid      14821084
holdout    14282445
Name: count, dtype: int64
tv_split_5
train      43351281
holdout    14334377
valid      14283978
Name: count, dtype: int64
Elapsed time :  110.26 seconds


In [14]:
df_a.shape

(71969636, 10)

In [15]:
df_a.columns

Index(['ID', 'pol', 'VIN', 'Date', 'superpolicy_id', 'tv_split_1',
       'tv_split_2', 'tv_split_3', 'tv_split_4', 'tv_split_5'],
      dtype='object')

In [33]:
df_a.drop('tv_split_5',axis = 1, inplace = True)

In [41]:
df_a.columns

Index(['ID', 'pol', 'VIN', 'Date', 'superpolicy_id', 'tv_split_2',
       'tv_split_3', 'tv_split_4', 'tv_split_1'],
      dtype='object')

In [18]:
df_a.head(10)

,ID,pol,VIN,Date,superpolicy_id,tv_split_1,tv_split_2,tv_split_3,tv_split_4,tv_split_5
0,1.0,EAA1345937,1J4GK48K42W273250,2017-01-01,1.0,train,holdout,holdout,train,valid
1,2.0,EAA1347517,1FMYU03122KA07434,2017-01-01,2.0,train,train,holdout,train,valid
2,3.0,EAA1347517,1FMYU03122KA07434,2018-01-01,2.0,train,train,holdout,train,valid
3,4.0,EAA1347848,2GTEC13V361342050,2017-01-01,4.0,train,train,train,train,train
4,5.0,EAA1347848,2GTEC13V361342050,2018-01-03,4.0,train,train,train,train,train
5,6.0,EAA1347848,1FTRW07643KA37402,2017-01-01,4.0,train,train,train,train,train
6,7.0,EAA1347848,1FTRW07643KA37402,2018-01-03,4.0,train,train,train,train,train
7,8.0,EAA1347851,1FAHP3EN1BW189557,2017-01-13,8.0,train,train,train,train,holdout
8,9.0,EAA1347851,1FAHP3EN1BW189557,2018-01-13,8.0,train,train,train,train,holdout
9,10.0,EAA1348082,3N1AB6AP4CL662610,2017-01-03,10.0,valid,train,holdout,holdout,train


In [11]:
if 0: 
    table = feather.read_table(feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)

In [70]:
# columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','claim_count_bi','claim_count_pd','claim_count_pip','claim_count_coll','claim_count_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','claim_count_med','VIN','Date','vc_Vehicle_Type','st']
columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','VIN','Date','vc_Vehicle_Type']
df2 = pd.read_feather(full_feather_file_path , columns = columns_to_load) 

In [71]:
df2.shape

(67451697, 22)

In [72]:
df2_1_superpolicy = pd.merge(df2,df_a, on = ['VIN','Date'], how = 'left')

In [73]:
df2_1_superpolicy.shape

(67641263, 30)

In [74]:
(df2_1_superpolicy.shape[0] - df2.shape[0])/df2_1_superpolicy.shape[0]

0.0028025201126123268

In [75]:
df2_1_superpolicy.head()

,ee_bi,ee_pd,ee_pip,ee_med,ee_coll,ee_comp,claim_count_bi,claim_count_pd,claim_count_pip,claim_count_coll,...,vc_Vehicle_Type,st,ID,pol,superpolicy_id,tv_split_1,tv_split_2,tv_split_3,tv_split_4,tv_split_5
0,0.342,0.342,0.342,0.342,0.342,0.342,0,0,0,0,...,SUV,MI,17009483.0,1000789936,17009483.0,train,train,train,train,valid
1,0.003,0.003,0.003,0.003,0.003,0.003,0,0,0,0,...,SUV,MI,16826626.0,1000346460,16071034.0,valid,train,train,train,train
2,0.999,0.999,0.000,0.999,0.999,0.999,0,0,0,0,...,SUV,MI,16494253.0,1000515508,16494252.0,train,train,valid,holdout,train
3,0.999,0.999,0.000,0.999,0.999,0.999,0,0,0,0,...,SUV,MI,16236590.0,1000368762,16116845.0,train,train,train,train,holdout
4,1.002,1.002,1.002,1.002,1.002,1.002,0,0,0,0,...,SUV,MI,16961077.0,1000752415,16961077.0,train,train,train,train,train


In [76]:
df2_1_superpolicy.columns

Index(['ee_bi', 'ee_pd', 'ee_pip', 'ee_med', 'ee_coll', 'ee_comp',
       'claim_count_bi', 'claim_count_pd', 'claim_count_pip',
       'claim_count_coll', 'claim_count_comp', 'incurred_raw_bi',
       'incurred_raw_pd', 'incurred_raw_pip', 'incurred_raw_coll',
       'incurred_raw_comp', 'incurred_raw_med', 'claim_count_med', 'VIN',
       'Date', 'vc_Vehicle_Type', 'st', 'ID', 'pol', 'superpolicy_id',
       'tv_split_1', 'tv_split_2', 'tv_split_3', 'tv_split_4', 'tv_split_5'],
      dtype='object')

In [77]:

df2_1_superpolicy.to_csv(file_df2_1_superpolicy,index = False )

In [ ]:
df1_1_superpolicy = pd.merge(df1_1,df_a, on = ['ID'], how = 'inner')

In [38]:
column_names = df1.columns.tolist()
columns_df = pd.DataFrame(column_names, columns = ['Column Names'] )
columns_df.to_csv(columns_file_path,index = False)

In [23]:
df1['ID'] = np.arange(1,len(df1)+1)
df1.shape

(71969636, 106)

In [18]:
df1.columns

Index(['VIN', 'zip', 'pol_py', 'companyid', 'st', 'pol_late_payments',
       'pol_veh_count', 'pol_driver_count', 'veh_limit_person_bi',
       'veh_limit_pd',
       ...
       'incurred_cap_comp', 'incurred_cap_tot', 'train', 'zip_geo',
       'geo_vacant_pct_ntile', 'geo_median_home_value_ntile',
       'geo_median_pop_age_ntile', 'geo_unemployment_rate_ntile',
       'geo_pop_density_ntile', 'ID'],
      dtype='object', length=106)

In [37]:
df2 = pd.merge(df_a,df1,on = 'ID',how = 'left' )

In [19]:
columns_to_write = ['ID','pol','VIN','Date',]

In [20]:
df2 = df1[columns_to_write]

In [21]:
df2.head()

,ID,pol,VIN,Date
0,1,EAA1345937,1J4GK48K42W273250,2017-01-01
1,2,EAA1347517,1FMYU03122KA07434,2017-01-01
2,3,EAA1347517,1FMYU03122KA07434,2018-01-01
3,4,EAA1347848,2GTEC13V361342050,2017-01-01
4,5,EAA1347848,2GTEC13V361342050,2018-01-03


In [8]:
#df1.head(1000).to_csv(csv_1000row_file_path,index = False)

In [23]:
df2.to_csv(csv_file_path, index = False)

In [ ]:
df2.to_csv(csv_allrow_file_path, index = False) 

In [25]:
outfile = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv'
df3 = pd.read_csv(outfile)

C:\Users\axs1\AppData\Local\Temp\ipykernel_5008\1159038369.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df3 = pd.read_csv(outfile)


In [26]:
df3.head()

,ID,pol,VIN,Date
0,1,EAA1345937,1J4GK48K42W273250,2017-01-01
1,2,EAA1347517,1FMYU03122KA07434,2017-01-01
2,3,EAA1347517,1FMYU03122KA07434,2018-01-01
3,4,EAA1347848,2GTEC13V361342050,2017-01-01
4,5,EAA1347848,2GTEC13V361342050,2018-01-03
